In [1]:
# ============================================================================
# STEP 1: INSTALL REQUIRED PACKAGES
# ============================================================================
!pip install -q langchain langchain-community langchain-core langchain-text-splitters
!pip install -q langchain-groq
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q beautifulsoup4 requests
print("\n Installation complete!")
print("="*70)


 Installation complete!


In [2]:
# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================
print("\n Importing libraries...")

import os
from getpass import getpass

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_core.chat_history import InMemoryChatMessageHistory

print(" All libraries imported successfully!")



 Importing libraries...


 All libraries imported successfully!


In [ ]:
  # Step  Set up API Key
import os
groq_api_key = os.environ.get("GROQ_API_KEY")
print("GROQ API key configured.")

GROQ API key configured.


In [5]:
# ============================================================================
# STEP 4: CREATE MULTI-SOURCE DATA
# ============================================================================
# Multiple URLs for better knowledge base

urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Large_language_model",
    "https://en.wikipedia.org/wiki/LangChain",
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Vector_database",
    "https://en.wikipedia.org/wiki/OpenAI"
]

from langchain_community.document_loaders import WebBaseLoader

docs = []

for url in urls:
    loader = WebBaseLoader(url)
    loaded_docs = loader.load()

    # ✅ attach source metadata
    for doc in loaded_docs:
        doc.metadata["source"] = url

    docs.extend(loaded_docs)

print(f"Loaded {len(docs)} documents from {len(urls)} URLs")

Loaded 6 documents from 6 URLs


In [6]:
# ============================================================================
# STEP 5: PROCESS AND INDEX ALL SOURCES
# ============================================================================
print("\n" + "="*70)
print(" PROCESSING DATA ")
print("="*70)

# Chunking
print("\nChunking documents...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} chunks")


# Embeddings
print("\nLoading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")


# Vector DB (FAISS)
print("\nCreating vector database...")
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector database created")


# Retriever
print("\nSetting up retriever...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever configured")


print("\nAll data indexed and ready!")
print("="*70)

print("\n All sources indexed and ready!")
print("="*70)


 PROCESSING DATA 

Chunking documents...
Created 753 chunks

Loading embedding model...


/tmp/ipykernel_6846/3913646453.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded

Creating vector database...
Vector database created

Setting up retriever...
Retriever configured

All data indexed and ready!

 All sources indexed and ready!


In [7]:
# ============================================================================
# STEP 6: INITIALIZE LLM AND MEMORY
# ============================================================================
print("\n" + "="*70)
print(" INITIALIZING AI COMPONENTS ")
print("="*70)

# LLM (Groq)
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("LLM ready (Groq)")


# Chat History (Modern Memory)
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()

print("Memory initialized")

print("\nAll AI components ready!")
print("="*70)


 INITIALIZING AI COMPONENTS 
LLM ready (Groq)
Memory initialized

All AI components ready!


In [9]:
# ============================================================================
# STEP 7: RAG PIPELINE
# ============================================================================
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def rag_pipeline(user_input):

    # STEP 1: Rewrite query using history
    standalone_question = contextualize_chain.invoke({
        "input": user_input,
        "chat_history": chat_history.messages
    })

    # STEP 2: Retrieve docs
    docs = retriever.get_relevant_documents(standalone_question)
    context = format_docs(docs)

    # STEP 3: Generate answer
    chain = qa_prompt | llm | StrOutputParser()

    response = chain.invoke({
        "input": user_input,
        "context": context,
        "chat_history": chat_history.messages
    })

    # STEP 4: Store history
    chat_history.add_user_message(user_input)
    chat_history.add_ai_message(response)

    return response

In [ ]:
# ============================================================================
# STEP 8: INTERACTIVE MODE
# ============================================================================
print("\n" + "="*70)
print(" INTERACTIVE MODE ")
print("="*70)
print("\nAsk anything (type 'exit' to quit)\n")

print("Commands:")
print(" 'sources'  - Show all data sources")
print(" 'clear'    - Clear conversation memory")
print(" 'exit'     - Exit\n")

while True:
    try:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("\nGoodbye 👋")
            break

        if user_input.lower() == "clear":
            chat_history.clear()
            print("🧹 Memory cleared\n")
            continue

        if user_input.lower() == "sources":
            print("\n📚 Sources used in this RAG system:\n")
            for i, url in enumerate(urls, 1):
                print(f"{i}. {url}")
            print()
            continue

        # 🔥 main pipeline
        response = rag_pipeline(user_input)
        print(f"Bot: {response}\n")

    except KeyboardInterrupt:
        print("\n\nChat interrupted. Goodbye!")
        break

    except Exception as e:
        print(f"\nError: {str(e)}\n")


 INTERACTIVE MODE 

Ask anything (type 'exit' to quit)

Commands:
 'sources'  - Show all data sources
 'clear'    - Clear conversation memory
 'exit'     - Exit

You: what is ai

Error: name 'contextualize_chain' is not defined

